# P03 — Privacidad, utilidad y aprendizaje distribuido

Velora Finance quiere obtener información útil de sus datos sin exponer más información de la necesaria.

Trabajaremos dos ideas:

1. **Differential Privacy (DP)**: añadir aleatoriedad calibrada para limitar cuánto puede influir un registro individual en un resultado publicado.
2. **Federated Learning (FL)**: entrenar de forma distribuida sin centralizar los datos brutos.

El objetivo no es implementar criptografía ni una plataforma federada de producción. El código está preparado para observar los **trade-offs** y las limitaciones de cada enfoque.


## 1. Preparación y datos


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import urllib.request

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

pd.set_option("display.max_columns", None)


In [ ]:
GENERATOR_URL = "https://raw.githubusercontent.com/pacosepulveda/ia-confiable-explicable-labs/main/resources/generate_velora_credit.py"

namespace = {"__name__": "velora_generator"}
with urllib.request.urlopen(GENERATOR_URL) as response:
    exec(response.read().decode("utf-8"), namespace)

df = namespace["generate_dataset"]()
print("Dataset cargado:", df.shape)


## 2. Una consulta aparentemente inocente

Para mantener el hilo de P01 y P02, Velora quiere publicar estadísticas sobre un **cohorte de auditoría de 600 solicitudes** obtenido con la misma partición reproducible utilizada anteriormente.

Con los datos completos del cohorte podemos calcular la tasa de impago exactamente.


In [ ]:
_, audit_index = train_test_split(
    df.index,
    test_size=0.20,
    random_state=42,
    stratify=df["default"]
)

audit_df = df.loc[audit_index].copy()
true_rate = audit_df["default"].mean()

print(f"Solicitudes del cohorte: {len(audit_df)}")
print(f"Tasa exacta de impago: {true_rate:.2%}")


Publicar un agregado reduce exposición respecto a publicar filas individuales, pero **agregación no equivale automáticamente a privacidad**.

Differential Privacy formaliza cuánto puede cambiar el resultado cuando añadimos o retiramos un único registro.


## 3. Mecanismo de Laplace sobre una tasa

Para una media de valores binarios y un tamaño `n` tratado como fijo, la sensibilidad es aproximadamente:

$$\Delta f = \frac{1}{n}$$

En un mecanismo de Laplace básico:

$$escala\ del\ ruido = \frac{\Delta f}{\epsilon}$$

Un `epsilon` menor implica una garantía de privacidad más fuerte, pero normalmente introduce más ruido.

> En esta demostración asumimos que el tamaño del conjunto consultado es conocido/fijo. En sistemas reales, el diseño de la consulta y del mecanismo requiere un análisis más cuidadoso.


In [ ]:
def dp_binary_rate(values, epsilon, rng):
    values = np.asarray(values)
    n = len(values)
    sensitivity = 1 / n
    noise = rng.laplace(loc=0.0, scale=sensitivity / epsilon)
    noisy_rate = values.mean() + noise
    return float(np.clip(noisy_rate, 0, 1)), float(noise)


## 4. Seleccionar el presupuesto de privacidad

Cambia únicamente `EPSILON` y observa el resultado y el error respecto a la tasa exacta.


In [ ]:
EPSILON = 1.00 #@param {type:"slider", min:0.10, max:5.00, step:0.10}
rng = np.random.default_rng(42)
private_rate, noise = dp_binary_rate(audit_df["default"].values, EPSILON, rng)

print(f"Epsilon        : {EPSILON:.2f}")
print(f"Tasa exacta    : {true_rate:.4%}")
print(f"Tasa con DP    : {private_rate:.4%}")
print(f"Ruido añadido  : {noise:+.6f}")
print(f"Error absoluto : {abs(private_rate - true_rate):.4%}")


### Qué significa este resultado

Una única ejecución puede resultar engañosa porque el mecanismo es aleatorio. Por eso repetiremos el experimento para comparar distintos valores de `epsilon`.


## 5. Privacidad vs. utilidad: muchas simulaciones


In [ ]:
epsilons = [0.10, 0.30, 1.00, 3.00, 5.00]
simulations = 500
rows = []

for epsilon in epsilons:
    errors = []
    for seed in range(simulations):
        rng = np.random.default_rng(seed)
        private_rate, _ = dp_binary_rate(audit_df["default"].values, epsilon, rng)
        errors.append(abs(private_rate - true_rate))
    rows.append({
        "epsilon": epsilon,
        "error_medio": np.mean(errors),
        "percentil_95_error": np.quantile(errors, 0.95),
    })

privacy_utility = pd.DataFrame(rows)
display(privacy_utility.style.format({
    "error_medio": "{:.4%}",
    "percentil_95_error": "{:.4%}",
}))


In [ ]:
plt.plot(privacy_utility["epsilon"], privacy_utility["error_medio"], marker="o")
plt.xlabel("Epsilon")
plt.ylabel("Error absoluto medio")
plt.title("Trade-off privacidad / utilidad")
plt.show()


### Interpretación

`epsilon` pequeño implica más ruido y normalmente menor utilidad; `epsilon` grande reduce el ruido pero debilita la garantía de privacidad. No existe un `epsilon` universalmente correcto.


## 6. El tamaño del grupo también importa

Publicamos la tasa de impago por grupo con el mismo `epsilon`. Un grupo más pequeño puede sufrir proporcionalmente más ruido.


In [ ]:
EPSILON_GRUPOS = 0.50 #@param {type:"slider", min:0.10, max:5.00, step:0.10}
rows = []

for i, (group, part) in enumerate(audit_df.groupby("customer_group")):
    exact = part["default"].mean()
    rng = np.random.default_rng(100 + i)
    private, noise = dp_binary_rate(part["default"].values, EPSILON_GRUPOS, rng)
    rows.append({
        "grupo": group,
        "n": len(part),
        "tasa_exacta": exact,
        "tasa_DP": private,
        "error_absoluto": abs(private - exact),
    })

group_dp = pd.DataFrame(rows).set_index("grupo")
display(group_dp.style.format({
    "tasa_exacta": "{:.3%}",
    "tasa_DP": "{:.3%}",
    "error_absoluto": "{:.3%}",
}))


### Conexión con fairness

Si un grupo tiene menos registros, sus estadísticas pueden ser menos estables bajo el mismo mecanismo. Privacidad y fairness no son dimensiones independientes.


## 7. El presupuesto se consume

En composición secuencial básica, `k` consultas con garantía `epsilon` pueden producir una cota acumulada aproximada:

$$\epsilon_{total} \le k \cdot \epsilon$$

Los sistemas reales pueden utilizar mecanismos y contabilidad más sofisticados.


In [ ]:
EPSILON_POR_CONSULTA = 0.50 #@param {type:"slider", min:0.10, max:2.00, step:0.10}
NUM_CONSULTAS = 10 #@param {type:"slider", min:1, max:50, step:1}
basic_total = EPSILON_POR_CONSULTA * NUM_CONSULTAS

print(f"Epsilon por consulta : {EPSILON_POR_CONSULTA:.2f}")
print(f"Número de consultas  : {NUM_CONSULTAS}")
print(f"Cota básica acumulada: ε <= {basic_total:.2f}")


# 8. Federated Learning: mover el cálculo hacia los datos

Differential Privacy y Federated Learning resuelven problemas diferentes. La siguiente es una **simulación local simplificada**: todo se ejecuta en el mismo notebook, pero separamos cuatro silos y hacemos que el agregador combine únicamente actualizaciones calculadas localmente.

No reproduce aislamiento real, secure aggregation ni toda la complejidad de una plataforma federada de producción.


In [ ]:
FED_FEATURES = ["debt_ratio", "late_payments_12m", "requested_amount", "annual_income", "savings_balance", "employment_months"]
fixed_scales = np.array([1.0, 7.0, 55000.0, 140000.0, 80000.0, 360.0])
X_fed = df[FED_FEATURES].to_numpy(dtype=float) / fixed_scales
X_fed = np.c_[np.ones(len(X_fed)), X_fed]
y_fed = df["default"].to_numpy(dtype=float)
client_ids = np.arange(len(df)) % 4
clients = []
for client_id in range(4):
    mask = client_ids == client_id
    clients.append((X_fed[mask], y_fed[mask]))

client_summary = pd.DataFrame({
    "cliente": [f"Silo {i+1}" for i in range(4)],
    "registros": [len(yc) for _, yc in clients],
    "tasa_impago": [yc.mean() for _, yc in clients],
})
display(client_summary.style.format({"tasa_impago": "{:.2%}"}))


In [ ]:
def sigmoid(z):
    z = np.clip(z, -30, 30)
    return 1 / (1 + np.exp(-z))

def local_gradient(X_local, y_local, weights):
    predictions = sigmoid(X_local @ weights)
    return X_local.T @ (predictions - y_local) / len(y_local)

def federated_gradient_descent(clients, rounds=200, learning_rate=0.8):
    weights = np.zeros(clients[0][0].shape[1])
    for _ in range(rounds):
        gradients = []
        sizes = []
        for X_local, y_local in clients:
            gradients.append(local_gradient(X_local, y_local, weights))
            sizes.append(len(y_local))
        total = sum(sizes)
        aggregated_gradient = sum(grad * (size / total) for grad, size in zip(gradients, sizes))
        weights -= learning_rate * aggregated_gradient
    return weights

federated_weights = federated_gradient_descent(clients)
fed_probability = sigmoid(X_fed @ federated_weights)
fed_auc = roc_auc_score(y_fed, fed_probability)

print(f"ROC-AUC de la simulación federada: {fed_auc:.3f}")
print(f"Parámetros intercambiados por cliente y ronda: {len(federated_weights)}")
print("Filas brutas enviadas al agregador: 0")


### Qué demuestra — y qué NO demuestra — esta simulación

La simulación ilustra la arquitectura lógica: cada silo calcula con sus datos y el agregador combina actualizaciones. Como todo se ejecuta en un único notebook, **no demuestra aislamiento técnico real**.

Federated Learning tampoco equivale a privacidad garantizada: actualizaciones y metadatos pueden filtrar información, y existen riesgos de inferencia, poisoning, clientes maliciosos y compromiso del agregador.


## 9. Comparar las dos ideas

| Técnica | Qué reduce principalmente | Qué NO garantiza por sí sola |
|---|---|---|
| Differential Privacy | influencia observable de registros individuales | exactitud, fairness, seguridad completa |
| Federated Learning | centralización de datos brutos | privacidad formal, ausencia de fugas, robustez |
| FL + DP | combina distribución y garantía estadística | que el sistema completo sea automáticamente confiable |


# Conclusiones de P03

- Agregar datos no equivale automáticamente a garantizar privacidad.
- Differential Privacy introduce un trade-off explícito entre privacidad y utilidad.
- `epsilon` no debe interpretarse sin contexto ni sin considerar composición.
- Los grupos pequeños pueden sufrir mayor pérdida de utilidad al aplicar ruido.
- Federated Learning evita centralizar datos brutos, pero no proporciona por sí mismo una garantía formal de privacidad.
- Combinar técnicas no elimina la necesidad de modelar amenazas, medir utilidad y auditar impactos.

El siguiente módulo cambiará de pregunta:

> Ya sabemos cómo se comporta el sistema. ¿Podemos explicar por qué produce una predicción concreta?
